# EXH Marketing Mix Model — Phase 2, Iteration 2
**Script 2 of 3: Diagnostics & Visualizations**

Loads all artifacts from Script 1 (trace, scalers, metadata, parquet) and produces the full set of model diagnostics and business charts. No model training occurs here — the trace is loaded from disk.

### Charts Produced
| Code | Description |
| --- | --- |
| D1 | Prior vs posterior — funnel hyperparameters |
| D3–D6 | Actual vs predicted, residuals, scatter |
| D7 | Adstock decay curves with CI by funnel |
| D10 | Response/saturation curves per funnel × original LOB |
| B1 | Stacked contribution decomposition (weekly) |
| B2–B5 | Monthly media by merged/original LOB |
| B6 | Annual bars — funnel decomposition |
| B9 | Annual bars — original LOB grouped |
| B10a/b | Aggregate response curve & marginal returns |

In [0]:
%pip install arviz scikit-learn scipy matplotlib openpyxl netcdf4 h5netcdf --quiet
dbutils.library.restartPython()

In [0]:
import os
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import beta as beta_dist, halfnorm
import arviz as az
from sklearn.metrics import mean_absolute_percentage_error, r2_score

## 1. Configuration — Paths & Brand Palette

In [0]:
ARTIFACT_PATH = "/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/MMM_EXH_Outcomes/"
SAVE_PATH     = ARTIFACT_PATH

CPX_NAVY  = "#002B5C"
CPX_RED   = "#E31837"
CPX_GOLD  = "#F5A800"
CPX_LGREY = "#F2F2F2"
CPX_DGREY = "#4A4A4A"


def save_fig(fig, filename):
    # """Save figure with Cineplex watermark to SAVE_PATH."""
    # # fig.text(
    # #     0.99, 0.01,
    # #     f"Cineplex Marketing Analytics  |  EXH MMM Phase 2 Iter 2  |  "
    # #     f"{pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}",
    # #     ha="right", va="bottom",
    # #     fontsize=6, color=CPX_DGREY, fontstyle="italic",
    # #     transform=fig.transFigure,
    # # )
    fpath = os.path.join(SAVE_PATH, filename)
    fig.savefig(fpath, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"  ✓ Saved → {filename}")

In [0]:
import arviz as az
trace = az.from_netcdf(f"{ARTIFACT_PATH}/trace_iter2_mixed_hier.nc")
# total number of free (unconstrained) parameters
print(sum(v.size // (trace.posterior.dims['chain'] * trace.posterior.dims['draw'])
          for v in trace.posterior.data_vars.values()))

In [0]:
import arviz as az
trace_iter2_mixed_hier = az.from_netcdf(
    f"{ARTIFACT_PATH}/trace_iter2_mixed_hier.nc"
)

ss = trace_iter2_mixed_hier.sample_stats
print("mean tree depth:", float(ss.tree_depth.mean()))
print("max tree depth :", int(ss.tree_depth.max()))
print("mean leapfrog  :", float(ss.n_steps.mean()))

## 2. Load Artifacts
Loads the trace, scalers, metadata, and parquet dataset produced by Script 1.

In [0]:
print("Loading artifacts...")

trace = az.from_netcdf(f"{ARTIFACT_PATH}trace_iter2_mixed_hier.nc")
post  = trace.posterior

with open(f"{ARTIFACT_PATH}scaler_y.pkl",              "rb") as f: scaler_y              = pickle.load(f)
with open(f"{ARTIFACT_PATH}scaler_X.pkl",              "rb") as f: scaler_X              = pickle.load(f)
with open(f"{ARTIFACT_PATH}spend_train_max_iter2.pkl", "rb") as f: spend_train_max_iter2 = pickle.load(f)
with open(f"{ARTIFACT_PATH}model_meta.pkl",            "rb") as f: model_meta            = pickle.load(f)
with open(f"{ARTIFACT_PATH}response_params.pkl",       "rb") as f: response_params       = pickle.load(f)

ACTIVE_COMBOS    = model_meta["ACTIVE_COMBOS"]
FUNNEL_COMBOS    = model_meta["FUNNEL_COMBOS"]
LOB_GROUP_MAP    = model_meta["LOB_GROUP_MAP"]
baseline_features = model_meta["baseline_features"]
target_col       = model_meta["target_col"]
n_train          = model_meta["n_train"]
n_val            = model_meta["n_val"]
n_test           = model_meta["n_test"]

df_full = pd.read_parquet(f"{ARTIFACT_PATH}df_full.parquet")

df_trainval = df_full[df_full["segment"].isin(["train", "val"])].copy().reset_index(drop=True)
df_trainval["year"] = df_trainval["week_date"].dt.year

train_dates = df_trainval.loc[df_trainval["segment"] == "train", "week_date"]
val_dates   = df_trainval.loc[df_trainval["segment"] == "val",   "week_date"]

actual_train_v = df_trainval.loc[df_trainval["segment"] == "train", target_col].values
actual_val_v   = df_trainval.loc[df_trainval["segment"] == "val",   target_col].values
pred_train_v   = df_trainval.loc[df_trainval["segment"] == "train", "pred"].values
pred_val_v     = df_trainval.loc[df_trainval["segment"] == "val",   "pred"].values

resid_train = actual_train_v - pred_train_v
resid_val   = actual_val_v   - pred_val_v

mape_train = mean_absolute_percentage_error(actual_train_v, pred_train_v) * 100
mape_val   = mean_absolute_percentage_error(actual_val_v,   pred_val_v)   * 100
r2_train   = r2_score(actual_train_v, pred_train_v)
r2_val     = r2_score(actual_val_v,   pred_val_v)

intercept_mean   = float(post["intercept"].values.ravel().mean())
baseline_orig    = intercept_mean * scaler_y.scale_[0] + scaler_y.mean_[0]
beta_mean        = post["beta"].values.reshape(-1, len(baseline_features)).mean(axis=0)
X_trainval       = scaler_X.transform(df_trainval[baseline_features].values)
mu_controls      = intercept_mean + X_trainval @ beta_mean
controls_effect  = (mu_controls - intercept_mean) * scaler_y.scale_[0]

print(f"  ✓ Trace loaded — {n_train + n_val + n_test} weeks total")
print(f"  ✓ df_full loaded — {len(df_full)} rows")

## 3. Colour Maps & Shared Constants
Brand palette, LOB colour assignments, year styling, and prior specifications used across all charts.

In [0]:
FUNNEL_COLORS = {
    "Awareness"    : CPX_NAVY,
    "Consideration": CPX_RED,
    "Conversion"   : CPX_GOLD,
}

LOB_COLORS = {
    "Awareness__Premium_Brand"    : "#1a3a5c",
    "Consideration__Premium_Brand": "#c0392b",
    "Consideration__GE"           : "#e74c3c",
    "Consideration__Prepaid"      : "#f1948a",
    "Conversion__Film_Marketing"  : "#f5a800",
    "Conversion__Premium_Brand"   : "#c47d00",
    "Conversion__Cineclub"        : "#ffd966",
}

LOB_LABELS = {
    "Awareness__Premium_Brand"    : "Awareness — Premium & Brand",
    "Consideration__Premium_Brand": "Consideration — Premium & Brand",
    "Consideration__GE"           : "Consideration — G&E",
    "Consideration__Prepaid"      : "Consideration — Prepaid",
    "Conversion__Film_Marketing"  : "Conversion — Film Marketing",
    "Conversion__Premium_Brand"   : "Conversion — Premium & Brand",
    "Conversion__Cineclub"        : "Conversion — Cineclub",
}

ORIG_LOB_PALETTE = {
    "Awareness__Premium Experiences - AVX"        : "#0d2137",
    "Awareness__Premium Experiences - VIP"        : "#1a3a5c",
    "Awareness__F&B"                              : "#2e5f8a",
    "Awareness__EXH - Brand"                      : "#4a85b5",
    "Consideration__Premium Experiences - AVX"    : "#7b0e0e",
    "Consideration__Premium Experiences - VIP"    : "#c0392b",
    "Consideration__F&B"                          : "#e74c3c",
    "Consideration__EXH - Brand"                  : "#f1948a",
    "Consideration__G&E"                          : "#922b21",
    "Consideration__Prepaid - B2B"                : "#fadbd8",
    "Consideration__Prepaid - B2C"                : "#f5b7b1",
    "Conversion__Premium Experiences - IMAX"      : "#7d5400",
    "Conversion__Film Marketing - Inform"         : "#c47d00",
    "Conversion__Performance - Convert & Upgrade" : "#f5a800",
    "Conversion__Premium Experiences - AVX"       : "#ffc300",
    "Conversion__Premium Experiences - VIP"       : "#ffd966",
    "Conversion__F&B"                             : "#ffe599",
    "Conversion__EXH - Brand"                     : "#fff2cc",
    "Conversion__Cineclub"                        : "#e6ac00",
}

ORIG_LOB_COLORS_BY_FUNNEL = {
    "Awareness": {
        "Premium Experiences - AVX": "#0d2137",
        "Premium Experiences - VIP": "#1a3a5c",
        "F&B"                      : "#2e5f8a",
        "EXH - Brand"              : "#4a85b5",
    },
    "Consideration": {
        "Premium Experiences - AVX": "#7b0e0e",
        "Premium Experiences - VIP": "#c0392b",
        "F&B"                      : "#e74c3c",
        "EXH - Brand"              : "#f1948a",
        "G&E"                      : "#922b21",
        "Prepaid - B2B"            : "#fadbd8",
        "Prepaid - B2C"            : "#f5b7b1",
    },
    "Conversion": {
        "Premium Experiences - IMAX"          : "#7d5400",
        "Film Marketing - Inform"             : "#c47d00",
        "Performance - Convert & Upgrade"     : "#f5a800",
        "Premium Experiences - AVX"           : "#ffc300",
        "Premium Experiences - VIP"           : "#ffd966",
        "F&B"                                 : "#ffe599",
        "EXH - Brand"                         : "#fff2cc",
        "Cineclub"                            : "#e6ac00",
    },
}

YEAR_LINESTYLES = {
    2023: {"color": "#2ecc71", "linestyle": "-",  "linewidth": 1.2},
    2024: {"color": "#e67e22", "linestyle": "--", "linewidth": 1.2},
    2025: {"color": "#9b59b6", "linestyle": ":",  "linewidth": 1.5},
}

YEAR_STYLES = {
    2023: {"color": "#2ecc71", "linestyle": "-",  "linewidth": 1.5, "marker": "o"},
    2024: {"color": "#e67e22", "linestyle": "--", "linewidth": 1.5, "marker": "s"},
    2025: {"color": "#9b59b6", "linestyle": ":",  "linewidth": 1.8, "marker": "D"},
}

ALPHA_MU_PRIOR = {
    "Awareness"    : {"alpha": 7, "beta": 2},
    "Consideration": {"alpha": 3, "beta": 3},
    "Conversion"   : {"alpha": 2, "beta": 5},
}

MU_GAMMA_PRIOR = {
    "Awareness"    : 0.5,
    "Consideration": 0.3,
    "Conversion"   : 0.6,
}

x_grid = np.linspace(0, 1.5, 300)

## 4. Saturation / Adstock Helper Functions (NumPy — for plotting)

In [0]:
def logistic_sat(x, lam):
    return 1 - np.exp(-lam * x)

def hill_sat(x, k, alpha_s):
    x_safe = np.maximum(x, 1e-6)
    return (x_safe ** alpha_s) / (x_safe ** alpha_s + k ** alpha_s)

def steady_state_adstock(s_norm, alpha):
    """Infinite-horizon adstock at constant normalised spend s_norm."""
    return s_norm / (1 - alpha)

def saturation_point_logistic(lam, threshold=0.95):
    return -np.log(1 - threshold) / lam

def saturation_point_hill(k, alpha_s, threshold=0.95):
    return k * (threshold / (1 - threshold)) ** (1 / alpha_s)

## 5. Original LOB Attribution
Each merged combo’s predicted contribution is split back to original LOBs in proportion to their weekly spend share within the group.

In [0]:
# Rebuild media_meta from raw media column names in the parquet
_media_candidates = [c for c in df_full.columns
                     if any(c.startswith(f"{f}_") for f in ["Awareness", "Consideration", "Conversion"])
                     and c not in ACTIVE_COMBOS
                     and not c.startswith("media_")
                     and not c.startswith("spend_")
                     and not c.startswith("contrib_")]

def _parse_media_col(col):
    parts = col.split("_")
    funnel = parts[0]
    known_channels = {"Digital", "Social", "SEM", "Other"}
    if parts[-1] in known_channels:
        channel = parts[-1]
        lob = " ".join(parts[1:-1])
    else:
        channel = "Unknown"
        lob = " ".join(parts[1:])
    return funnel, lob, channel

media_meta = pd.DataFrame([
    {"col_name": c, "funnel": _parse_media_col(c)[0], "lob": _parse_media_col(c)[1]}
    for c in _media_candidates
])

# Build proper spend columns from raw media columns on both DataFrames
for target_df in [df_trainval, df_full]:
    for funnel in ["Awareness", "Consideration", "Conversion"]:
        for lob_group, orig_lobs in LOB_GROUP_MAP.items():
            combo = f"{funnel}__{lob_group}"
            if combo not in ACTIVE_COMBOS:
                continue
            for orig_lob in orig_lobs:
                raw_cols = media_meta.loc[
                    (media_meta["funnel"] == funnel) &
                    (media_meta["lob"] == orig_lob),
                    "col_name"
                ].tolist()
                col_name = f"spend_{funnel}__{orig_lob}"
                target_df[col_name] = target_df[raw_cols].sum(axis=1).values if raw_cols else 0.0

ORIG_LOB_CONTRIB = {}

for target_df in [df_trainval, df_full]:
    for funnel in ["Awareness", "Consideration", "Conversion"]:
        for lob_group, orig_lobs in LOB_GROUP_MAP.items():
            combo = f"{funnel}__{lob_group}"
            if combo not in ACTIVE_COMBOS:
                continue
            media_col     = f"media_{combo}"
            combo_contrib = target_df[media_col].values
            combo_spend   = target_df[combo].values if combo in target_df.columns else np.zeros(len(target_df))

            for orig_lob in orig_lobs:
                spend_col  = f"spend_{funnel}__{orig_lob}"
                orig_spend = target_df[spend_col].values
                share      = np.where(combo_spend > 0, orig_spend / combo_spend, 0.0)
                contrib_col = f"contrib_{funnel}__{orig_lob}"
                target_df[contrib_col] = combo_contrib * share
                if target_df is df_trainval:
                    ORIG_LOB_CONTRIB[contrib_col] = {
                        "funnel"  : funnel,
                        "orig_lob": orig_lob,
                        "combo"   : combo,
                    }


print(f"  ✓ ORIG_LOB_CONTRIB built — {len(ORIG_LOB_CONTRIB)} entries (df_trainval + df_full)")

## 6. Chart D1 — Hyperparameter Prior vs Posterior
3×4 grid: rows = funnels, cols = mu_alpha, kappa_alpha, mu_gamma, Hill hyperpriors.

In [0]:
print("\nGenerating diagnostic charts...")

x_alpha  = np.linspace(0, 1,  300)
x_gamma  = np.linspace(0, 2,  300)
x_k      = np.linspace(0, 2,  300)
x_alphas = np.linspace(0, 6,  300)
x_kappa  = np.linspace(0, 10, 300)

fig, axes = plt.subplots(3, 4, figsize=(22, 14))

for row, funnel in enumerate(["Awareness", "Consideration", "Conversion"]):
    color = FUNNEL_COLORS[funnel]
    ap    = ALPHA_MU_PRIOR[funnel]

    ax = axes[row, 0]
    ax.plot(x_alpha, beta_dist.pdf(x_alpha, ap["alpha"], ap["beta"]),
            color=CPX_DGREY, linewidth=1.5, linestyle="--", label="Prior")
    ax.hist(post[f"mu_alpha_{funnel}"].values.ravel(), bins=60, density=True,
            color=color, alpha=0.6, label="Posterior")
    ax.set_title(f"{funnel} — mu_alpha (pooled decay mean)", fontweight="bold", color=CPX_NAVY)
    ax.set_xlabel("mu_alpha"); ax.legend(fontsize=7)
    ax.grid(False); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    ax = axes[row, 1]
    ax.plot(x_kappa, halfnorm.pdf(x_kappa, scale=2),
            color=CPX_DGREY, linewidth=1.5, linestyle="--", label="Prior")
    ax.hist(post[f"kappa_alpha_{funnel}"].values.ravel(), bins=60, density=True,
            color=color, alpha=0.6, label="Posterior")
    ax.set_title(f"{funnel} — kappa_alpha (pooling concentration)", fontweight="bold", color=CPX_NAVY)
    ax.set_xlabel("kappa_alpha"); ax.legend(fontsize=7)
    ax.grid(False); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    ax = axes[row, 2]
    ax.plot(x_gamma, halfnorm.pdf(x_gamma, scale=MU_GAMMA_PRIOR[funnel]),
            color=CPX_DGREY, linewidth=1.5, linestyle="--", label="Prior")
    ax.hist(post[f"mu_gamma_{funnel}"].values.ravel(), bins=60, density=True,
            color=color, alpha=0.6, label="Posterior")
    ax.set_title(f"{funnel} — mu_gamma (pooled effect mean)", fontweight="bold", color=CPX_NAVY)
    ax.set_xlabel("mu_gamma"); ax.legend(fontsize=7)
    ax.grid(False); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    ax = axes[row, 3]
    if funnel == "Consideration":
        ax.plot(x_k, halfnorm.pdf(x_k, scale=0.5),
                color=CPX_DGREY, linewidth=1.5, linestyle="--", label="Prior (mu_k)")
        ax.hist(post["mu_k_Consideration"].values.ravel(), bins=60, density=True,
                color=color, alpha=0.6, label="Posterior (mu_k)")
        ax2 = ax.twinx()
        ax2.plot(x_alphas, halfnorm.pdf(x_alphas, scale=1.5),
                 color=CPX_DGREY, linewidth=1.5, linestyle=":", label="Prior (mu_alpha_s)")
        ax2.hist(post["mu_alpha_s_Consideration"].values.ravel(), bins=60,
                 density=True, color=color, alpha=0.3, label="Posterior (mu_alpha_s)")
        ax2.axvline(1.0, color=CPX_DGREY, linewidth=1, linestyle=":", label="α_s=1 (hyperbolic)")
        ax2.set_ylabel("mu_alpha_s density", fontsize=7); ax2.legend(fontsize=6, loc="upper right")
        ax.set_title("Consideration — mu_k & mu_alpha_s (Hill)", fontweight="bold", color=CPX_NAVY)
        ax.set_xlabel("value"); ax.legend(fontsize=6, loc="upper left")
    else:
        ax.text(0.5, 0.5, "No Hill\nhyperpriors", ha="center", va="center",
                transform=ax.transAxes, fontsize=10, color=CPX_DGREY, fontstyle="italic")
        ax.set_title(f"{funnel} — Hill hyperpriors", fontweight="bold", color=CPX_NAVY)
    ax.grid(False); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

fig.suptitle(
    "Prior vs Posterior — Hyperparameters (Final Model: Iter 2 Mixed Hierarchical)\n"
    "Rows = Funnels  |  Cols = mu_alpha, kappa_alpha, mu_gamma, Hill hyperpriors",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
plt.tight_layout()
save_fig(fig, "I2_D1_Hyperparameter_Posteriors.png")
plt.show()

## 7. Chart D7 — Adstock Decay Curves by Funnel
16-week decay with 95% credible intervals.

In [0]:
WEEKS_OUT = 16
weeks     = np.arange(WEEKS_OUT)

fig, ax = plt.subplots(figsize=(10, 5))

for funnel in ["Awareness", "Consideration", "Conversion"]:
    samples    = post[f"mu_alpha_{funnel}"].values.ravel()
    alpha_mean = float(samples.mean())
    alpha_lo   = float(np.quantile(samples, 0.025))
    alpha_hi   = float(np.quantile(samples, 0.975))

    decay_mean = np.array([alpha_mean ** t for t in range(WEEKS_OUT)])
    decay_lo   = np.array([alpha_lo   ** t for t in range(WEEKS_OUT)])
    decay_hi   = np.array([alpha_hi   ** t for t in range(WEEKS_OUT)])

    ax.plot(weeks, decay_mean, color=FUNNEL_COLORS[funnel], linewidth=2,
            label=f"{funnel} (μ_α={alpha_mean:.3f})")
    ax.fill_between(weeks, decay_lo, decay_hi, color=FUNNEL_COLORS[funnel], alpha=0.15)

ax.axhline(0.05, color=CPX_DGREY, linewidth=0.8, linestyle="--", alpha=0.6, label="5% threshold")
ax.set_xlabel("Weeks after spend"); ax.set_ylabel("Remaining adstock effect (relative)")
ax.set_title(
    "Adstock Decay Curves by Funnel (Final Model)\n"
    "Shaded = 95% credible interval  |  Dashed = 5% threshold",
    fontsize=11, fontweight="bold", color=CPX_NAVY,
)
ax.legend(fontsize=9); ax.grid(False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "I2_D7_Adstock_Decay_Funnel.png")
plt.show()

## 8. Charts D3–D6 — Validation Diagnostics
Actual vs predicted (full + val window), residual structure, and predicted-vs-actual scatter.

In [0]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_trainval["week_date"], df_trainval[target_col] / 1e6,
        color="black", linewidth=1.5, label="Actual")
ax.plot(df_trainval["week_date"], df_trainval["pred"] / 1e6,
        color=CPX_NAVY, linewidth=1.5, linestyle="--", label="Predicted")
ax.plot(df_trainval["week_date"], df_trainval["controls_pred"] / 1e6,
        color=CPX_DGREY, linewidth=1.0, linestyle=":", alpha=0.7, label="Baseline (controls only)")
ax.axvspan(val_dates.iloc[0], val_dates.iloc[-1], alpha=0.08, color=CPX_RED, label="Validation period")
ax.set_ylabel("Weekly Attendance (M)")
ax.set_title(
    f"Actual vs Predicted (Final Model)  —  "
    f"Train MAPE {mape_train:.1f}%  R² {r2_train:.3f}  |  Val MAPE {mape_val:.1f}%  R² {r2_val:.3f}",
    fontsize=11, fontweight="bold", color=CPX_NAVY,
)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}M"))
ax.legend(fontsize=8, ncol=2); ax.grid(False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "I2_D3_Actual_vs_Predicted.png")
plt.show()

In [0]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(val_dates.values, actual_val_v / 1e6, label="Actual", color="steelblue", linewidth=2)
ax.plot(val_dates.values, pred_val_v   / 1e6, label="Predicted", color="darkorange", linewidth=2, linestyle="--")
ax.fill_between(val_dates.values, actual_val_v / 1e6, pred_val_v / 1e6,
                where=(actual_val_v >= pred_val_v), alpha=0.2, color="green", label="Under-predicted")
ax.fill_between(val_dates.values, actual_val_v / 1e6, pred_val_v / 1e6,
                where=(actual_val_v < pred_val_v),  alpha=0.2, color="red",   label="Over-predicted")
ax.set_ylabel("Weekly Attendance (M)")
ax.set_title("Validation Window — Actual vs Predicted", fontsize=11, fontweight="bold", color=CPX_NAVY)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.2f}M"))
ax.legend(fontsize=9); ax.grid(False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "I2_D4_Val_Actual_vs_Predicted.png")
plt.show()

In [0]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train_dates.values, resid_train / 1e3, color=CPX_NAVY, linewidth=1, label="Train residual")
ax.bar(val_dates.values, resid_val / 1e3, color=CPX_RED, alpha=0.7, width=5, label="Val residual")
ax.axhline(0, color="black", linewidth=0.8)
ax.axhline( resid_train.std() / 1e3, color=CPX_DGREY, linewidth=0.8, linestyle="--", alpha=0.6,
           label=f"±1σ train ({resid_train.std()/1e3:,.0f}K)")
ax.axhline(-resid_train.std() / 1e3, color=CPX_DGREY, linewidth=0.8, linestyle="--", alpha=0.6)
ax.axhline(resid_val.mean() / 1e3, color=CPX_RED, linewidth=1.2, linestyle="--",
           label=f"Val mean ({resid_val.mean()/1e3:,.0f}K)")
ax.set_ylabel("Residual (000s)")
ax.set_title("Residuals Over Time — Target: Random, No Structure", fontsize=11, fontweight="bold", color=CPX_NAVY)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}K"))
ax.legend(fontsize=8, ncol=2); ax.grid(False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "I2_D5_Residuals_Over_Time.png")
plt.show()

In [0]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(actual_train_v / 1e6, pred_train_v / 1e6, alpha=0.5, s=25, color=CPX_NAVY, label="Train")
ax.scatter(actual_val_v   / 1e6, pred_val_v   / 1e6, alpha=0.9, s=60, color=CPX_RED, marker="D", label="Val")
lim_lo = min(actual_train_v.min(), pred_train_v.min()) / 1e6
lim_hi = max(actual_train_v.max(), pred_train_v.max()) / 1e6
ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], "k--", linewidth=1, label="Perfect fit")
ax.set_xlabel("Actual Attendance (M)"); ax.set_ylabel("Predicted Attendance (M)")
ax.set_title("Predicted vs Actual Scatter", fontsize=11, fontweight="bold", color=CPX_NAVY)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}M"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}M"))
ax.legend(fontsize=8); ax.grid(False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "I2_D6_Scatter_Predicted_vs_Actual.png")
plt.show()

## 9. Chart B1 — Weekly Contribution Decomposition (Stacked Area)
Stacked: Intercept → Controls → Media by Funnel. Black line = actual attendance.

In [0]:
print("\nGenerating business charts...")

fig, ax = plt.subplots(figsize=(18, 7))
ax.fill_between(df_trainval["week_date"], 0, baseline_orig / 1e6,
                alpha=0.9, color=CPX_LGREY,
                label=f"Intercept baseline ({baseline_orig/1e6:.2f}M avg)")
ax.fill_between(df_trainval["week_date"],
                baseline_orig / 1e6,
                (baseline_orig + controls_effect) / 1e6,
                alpha=0.7, color=CPX_DGREY, label="Controls effect")

stack_top = (baseline_orig + controls_effect).copy()
for funnel, combos in FUNNEL_COMBOS.items():
    media_f = df_trainval[f"media_{funnel}"].values
    ax.fill_between(df_trainval["week_date"], stack_top / 1e6, (stack_top + media_f) / 1e6,
                    alpha=0.85, color=FUNNEL_COLORS[funnel], label=f"{funnel} media")
    stack_top += media_f

ax.plot(df_trainval["week_date"], df_trainval[target_col] / 1e6,
        color="black", linewidth=1.5, label="Actual", zorder=5)
ax.axvspan(val_dates.iloc[0], val_dates.iloc[-1], alpha=0.06, color=CPX_RED)
ax.annotate("← Validation", xy=(val_dates.iloc[0], ax.get_ylim()[1] * 0.95),
            fontsize=8, color=CPX_RED, fontstyle="italic")
ax.set_ylabel("Weekly Attendance (M)", color=CPX_NAVY)
ax.set_title(
    "EXH MMM — Attendance Decomposition by Component\n"
    "Stacked: Intercept → Controls → Media by Funnel  |  Line = Actual",
    fontsize=12, fontweight="bold", color=CPX_NAVY,
)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}M"))
ax.legend(fontsize=8, ncol=4, loc="upper left"); ax.grid(False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "I2_B1_Contribution_Decomposition_Final.png")
plt.show()

## 10. Chart B6 — Annual Decomposition by Funnel

In [0]:
def bar_label(ax, x_pos, bottom, height, pct, color, min_height=0.5):
    """Add percentage label inside a bar segment."""
    if abs(height) / 1e6 < min_height:
        return
    light_colors = {CPX_LGREY, CPX_GOLD}
    txt_color = "black" if color in light_colors else "white"
    ax.text(x_pos, (bottom + height / 2) / 1e6, f"{pct:.0f}%",
            ha="center", va="center", fontsize=8, fontweight="bold", color=txt_color,
            path_effects=[pe.withStroke(linewidth=2,
                          foreground="white" if txt_color == "black" else "black")])

df_full["year"] = df_full["week_date"].dt.year

agg_dict = {target_col: "sum"}
for funnel in FUNNEL_COMBOS:
    agg_dict[f"media_{funnel}"] = "sum"
for col in ORIG_LOB_CONTRIB:
    agg_dict[col] = "sum"

annual       = df_full[df_full["year"] != 2026].groupby("year").agg(agg_dict).round(0)
years_avail  = sorted(annual.index.tolist())
actual_vals  = annual.loc[years_avail, target_col].values
x            = np.arange(len(years_avail))
w            = 0.6

awareness_vals     = annual.loc[years_avail, "media_Awareness"].values
consideration_vals = annual.loc[years_avail, "media_Consideration"].values
conversion_vals    = annual.loc[years_avail, "media_Conversion"].values
baseline_vals      = actual_vals - awareness_vals - consideration_vals - conversion_vals

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x, baseline_vals / 1e6,     width=w, color=CPX_LGREY, alpha=0.9, edgecolor="white", linewidth=0.5, label="Baseline (non-media)")
ax.bar(x, awareness_vals / 1e6,    width=w, bottom=baseline_vals / 1e6, color=CPX_NAVY, alpha=0.9, edgecolor="white", linewidth=0.5, label="Awareness")
ax.bar(x, consideration_vals / 1e6, width=w, bottom=(baseline_vals + awareness_vals) / 1e6, color=CPX_RED, alpha=0.9, edgecolor="white", linewidth=0.5, label="Consideration")
ax.bar(x, conversion_vals / 1e6,   width=w, bottom=(baseline_vals + awareness_vals + consideration_vals) / 1e6, color=CPX_GOLD, alpha=0.9, edgecolor="white", linewidth=0.5, label="Conversion")

for i, (v, b) in enumerate(zip(baseline_vals, np.zeros(len(years_avail)))):
    bar_label(ax, x[i], b, v, v / actual_vals[i] * 100, CPX_LGREY)
for i, (v, b) in enumerate(zip(awareness_vals, baseline_vals)):
    bar_label(ax, x[i], b, v, v / actual_vals[i] * 100, CPX_NAVY, 1.5)
for i, (v, b) in enumerate(zip(consideration_vals, baseline_vals + awareness_vals)):
    bar_label(ax, x[i], b, v, v / actual_vals[i] * 100, CPX_RED, 1.5)
for i, (v, b) in enumerate(zip(conversion_vals, baseline_vals + awareness_vals + consideration_vals)):
    bar_label(ax, x[i], b, v, v / actual_vals[i] * 100, CPX_GOLD, 1.5)

ax.scatter(x, actual_vals / 1e6, color="black", zorder=5, s=60, marker="D", label="Actual")
ax.set_xticks(np.arange(len(years_avail))); ax.set_xticklabels(years_avail)
ax.set_ylabel("Annual Attendance (in Millions)")
ax.set_title("Annual Attendance Decomposition by Funnel", fontsize=12, fontweight="bold", color=CPX_NAVY)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}M"))
ax.legend(fontsize=8, ncol=2); ax.grid(False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "I2_B6_Annual_Decomposition_Funnel.png")
plt.show()

## 11. Monthly LOB Charts (B2, B3, B4, B5)
Monthly media contribution stacked bars — merged LOBs, original LOBs, and per-funnel breakdowns.

In [0]:
df_trainval["month"]    = df_trainval["week_date"].dt.to_period("M")
monthly_agg_dict        = {target_col: "sum"}
for combo in ACTIVE_COMBOS:
    monthly_agg_dict[f"media_{combo}"] = "sum"
for col in ORIG_LOB_CONTRIB:
    monthly_agg_dict[col] = "sum"

monthly             = df_trainval.groupby("month").agg(monthly_agg_dict).reset_index()
monthly["month_dt"] = monthly["month"].dt.to_timestamp()

val_start   = df_trainval.loc[df_trainval["segment"] == "val", "week_date"].dt.to_period("M").min().to_timestamp()
val_months  = monthly[monthly["month_dt"] >= val_start].index
month_labels = [m.strftime("%b %Y") for m in monthly["month_dt"]]
x_m          = np.arange(len(monthly))
w_m          = 0.65


def add_val_shade(ax):
    if len(val_months) > 0:
        ax.axvspan(val_months[0] - 0.5, val_months[-1] + 0.5, alpha=0.06, color=CPX_RED)
        ax.annotate("← Validation", xy=(val_months[0], ax.get_ylim()[1] * 0.97),
                    fontsize=8, color=CPX_RED, fontstyle="italic")

## 14. Charts B10 — Total Media Response Curve & Marginal Returns
Aggregate response curve (steady-state adstock, 2025 LOB budget shares held constant) and marginal attendees per $1 additional weekly spend.

In [0]:
print("\nGenerating aggregate response and marginal return curves...")


def total_media_attendance(weekly_budget_alloc, params):
    """Estimate annual media-driven attendance given a weekly budget allocation dict."""
    total = 0.0
    for combo, weekly_spend in weekly_budget_alloc.items():
        funnel    = combo.split("__")[0]
        max_spend = spend_train_max_iter2[combo]
        alpha     = params[f"alpha_{combo}"]
        gamma     = params[f"gamma_{combo}"]
        s_norm    = weekly_spend / max_spend
        adstock   = steady_state_adstock(s_norm, alpha)
        if funnel == "Consideration":
            sat = hill_sat(adstock, params[f"k_{combo}"], params[f"alpha_s_{combo}"])
        else:
            sat = logistic_sat(adstock, params[f"lam_{combo}"])
        total += gamma * sat * scaler_y.scale_[0] * 52
    return total


params_mean = {}
for combo in ACTIVE_COMBOS:
    funnel = combo.split("__")[0]
    params_mean[f"alpha_{combo}"] = float(post[f"alpha_{combo}"].values.ravel().mean())
    params_mean[f"gamma_{combo}"] = float(post[f"gamma_{combo}"].values.ravel().mean())
    if funnel == "Consideration":
        params_mean[f"k_{combo}"]       = float(post[f"k_{combo}"].values.ravel().mean())
        params_mean[f"alpha_s_{combo}"] = float(post[f"alpha_s_{combo}"].values.ravel().mean())
    else:
        params_mean[f"lam_{combo}"] = float(post[f"lam_{combo}"].values.ravel().mean())

df_2025         = df_full[df_full["year"] == 2025]
total_2025_wk   = sum(df_2025[c].mean() for c in ACTIVE_COMBOS)
budget_shares   = {c: df_2025[c].mean() / total_2025_wk for c in ACTIVE_COMBOS}

total_2025_annual = total_2025_wk * 52
x_annual   = np.linspace(52, total_2025_annual * 10, 300)
curve_mean = np.array([
    total_media_attendance({c: (b / 52) * budget_shares[c] for c in ACTIVE_COMBOS}, params_mean)
    for b in x_annual
])

n_samples  = 100
flat_len   = post[f"gamma_{ACTIVE_COMBOS[0]}"].values.ravel().shape[0]
sample_idx = np.linspace(0, flat_len - 1, n_samples, dtype=int)

curve_samples = []
for i in sample_idx:
    p = {}
    for combo in ACTIVE_COMBOS:
        funnel = combo.split("__")[0]
        p[f"alpha_{combo}"] = float(post[f"alpha_{combo}"].values.ravel()[i])
        p[f"gamma_{combo}"] = float(post[f"gamma_{combo}"].values.ravel()[i])
        if funnel == "Consideration":
            p[f"k_{combo}"]       = float(post[f"k_{combo}"].values.ravel()[i])
            p[f"alpha_s_{combo}"] = float(post[f"alpha_s_{combo}"].values.ravel()[i])
        else:
            p[f"lam_{combo}"] = float(post[f"lam_{combo}"].values.ravel()[i])
    curve_samples.append(np.array([
        total_media_attendance({c: (b / 52) * budget_shares[c] for c in ACTIVE_COMBOS}, p)
        for b in x_annual
    ]))

curve_samples = np.array(curve_samples)
curve_lo      = np.quantile(curve_samples, 0.055, axis=0)
curve_hi      = np.quantile(curve_samples, 0.945, axis=0)

year_points = {}
for year in [2023, 2024, 2025]:
    df_yr        = df_full[df_full["year"] == year]
    annual_spend = sum(df_yr[c].sum() for c in ACTIVE_COMBOS)
    annual_att   = total_media_attendance(
        {c: df_yr[c].mean() for c in ACTIVE_COMBOS}, params_mean)
    year_points[year] = {"annual_spend": annual_spend, "annual_att": annual_att}

# Marginal returns & inflection point
dx            = x_annual[1] - x_annual[0]
marginal_mean = np.gradient(curve_mean, dx)
second_deriv  = np.gradient(marginal_mean, dx)
inflection_idx = next((i for i in range(1, len(second_deriv))
                       if second_deriv[i - 1] > 0 and second_deriv[i] <= 0), None)

# 95% saturation point
sat_95_idx   = np.argmin(np.abs(curve_mean - curve_mean[-1] * 0.95))
sat_95_spend = x_annual[sat_95_idx]
sat_95_att   = curve_mean[sat_95_idx]

if inflection_idx:
    inf_spend = x_annual[inflection_idx]
    inf_att   = curve_mean[inflection_idx]

In [0]:
print("\nGenerating aggregate response and marginal return curves...")


def total_media_attendance(weekly_budget_alloc, params):
    """
    Estimate annual media-driven attendance given a weekly budget allocation dict.
    Uses steady-state adstock (infinite horizon at constant weekly spend).

    Parameters
    ----------
    weekly_budget_alloc : dict  combo → weekly $ spend
    params              : dict  posterior parameters (alpha, gamma, lam/k/alpha_s)

    Returns
    -------
    float — estimated annual media attendance contribution
    """
    total = 0.0
    for combo, weekly_spend in weekly_budget_alloc.items():
        funnel    = combo.split("__")[0]
        max_spend = spend_train_max_iter2[combo]
        alpha     = params[f"alpha_{combo}"]
        gamma     = params[f"gamma_{combo}"]
        s_norm    = weekly_spend / max_spend
        adstock   = steady_state_adstock(s_norm, alpha)
        if funnel == "Consideration":
            sat = hill_sat(adstock, params[f"k_{combo}"], params[f"alpha_s_{combo}"])
        else:
            sat = logistic_sat(adstock, params[f"lam_{combo}"])
        total += gamma * sat * scaler_y.scale_[0] * 52
    return total


# Extract posterior means
params_mean = {}
for combo in ACTIVE_COMBOS:
    funnel = combo.split("__")[0]
    params_mean[f"alpha_{combo}"] = float(post[f"alpha_{combo}"].values.ravel().mean())
    params_mean[f"gamma_{combo}"] = float(post[f"gamma_{combo}"].values.ravel().mean())
    if funnel == "Consideration":
        params_mean[f"k_{combo}"]       = float(post[f"k_{combo}"].values.ravel().mean())
        params_mean[f"alpha_s_{combo}"] = float(post[f"alpha_s_{combo}"].values.ravel().mean())
    else:
        params_mean[f"lam_{combo}"] = float(post[f"lam_{combo}"].values.ravel().mean())

# 2025 budget shares (hold allocation mix constant as we vary total budget)
df_2025          = df_full[df_full["year"] == 2025]
total_2025_wk    = sum(df_2025[c].mean() for c in ACTIVE_COMBOS)
total_2025_annual = total_2025_wk * 52
budget_shares    = {c: df_2025[c].mean() / total_2025_wk for c in ACTIVE_COMBOS}

# X axis is annual spend; pass weekly equivalent into the response function
x_annual   = np.linspace(52, total_2025_annual * 10, 300)
curve_mean = np.array([
    total_media_attendance({c: (b / 52) * budget_shares[c] for c in ACTIVE_COMBOS}, params_mean)
    for b in x_annual
])

# 89% CI from 100 posterior samples
n_samples  = 100
flat_len   = post[f"gamma_{ACTIVE_COMBOS[0]}"].values.ravel().shape[0]
sample_idx = np.linspace(0, flat_len - 1, n_samples, dtype=int)

curve_samples = []
for i in sample_idx:
    p = {}
    for combo in ACTIVE_COMBOS:
        funnel = combo.split("__")[0]
        p[f"alpha_{combo}"] = float(post[f"alpha_{combo}"].values.ravel()[i])
        p[f"gamma_{combo}"] = float(post[f"gamma_{combo}"].values.ravel()[i])
        if funnel == "Consideration":
            p[f"k_{combo}"]       = float(post[f"k_{combo}"].values.ravel()[i])
            p[f"alpha_s_{combo}"] = float(post[f"alpha_s_{combo}"].values.ravel()[i])
        else:
            p[f"lam_{combo}"] = float(post[f"lam_{combo}"].values.ravel()[i])
    curve_samples.append(np.array([
        total_media_attendance({c: (b / 52) * budget_shares[c] for c in ACTIVE_COMBOS}, p)
        for b in x_annual
    ]))

curve_samples = np.array(curve_samples)
curve_lo      = np.quantile(curve_samples, 0.055, axis=0)
curve_hi      = np.quantile(curve_samples, 0.945, axis=0)

# Year reference points — use annual_spend directly on the X axis
year_points = {}
for year in [2023, 2024, 2025]:
    df_yr        = df_full[df_full["year"] == year]
    annual_spend = sum(df_yr[c].sum() for c in ACTIVE_COMBOS)
    annual_att   = total_media_attendance(
        {c: df_yr[c].mean() for c in ACTIVE_COMBOS}, params_mean)
    year_points[year] = {"annual_spend": annual_spend, "annual_att": annual_att}

# Marginal returns & inflection point
# Gradient is per $ of annual spend; divide by 52 gives per $ of weekly spend
dx            = x_annual[1] - x_annual[0]
marginal_mean = np.gradient(curve_mean, dx)
second_deriv  = np.gradient(marginal_mean, dx)
inflection_idx = next((i for i in range(1, len(second_deriv))
                       if second_deriv[i - 1] > 0 and second_deriv[i] <= 0), None)

# 95% saturation point
sat_95_idx   = np.argmin(np.abs(curve_mean - curve_mean[-1] * 0.95))
sat_95_spend = x_annual[sat_95_idx]   # annual $
sat_95_att   = curve_mean[sat_95_idx]

# B10a: Total response curve — both axes annual
fig, ax = plt.subplots(figsize=(16, 9))
ax.plot(x_annual / 1e6, curve_mean / 1e6, color=CPX_NAVY, linewidth=2.5,
        label="Total media response (posterior mean)")
ax.fill_between(x_annual / 1e6, curve_lo / 1e6, curve_hi / 1e6,
                color=CPX_NAVY, alpha=0.12, label="89% credible interval")

if inflection_idx:
    inf_spend = x_annual[inflection_idx]
    inf_att   = curve_mean[inflection_idx]
    ax.axvline(inf_spend / 1e6, color=CPX_DGREY, linewidth=1.0, linestyle="--", alpha=0.7)
    ax.scatter([inf_spend / 1e6], [inf_att / 1e6], color=CPX_DGREY, s=80, zorder=6, marker="^")
    ax.annotate(f"Inflection\n${inf_spend/1e6:.1f}M/yr",
                xy=(inf_spend / 1e6, inf_att / 1e6), xytext=(12, -40), textcoords="offset points",
                fontsize=8, color=CPX_DGREY, fontweight="bold",
                arrowprops=dict(arrowstyle="->", color=CPX_DGREY, lw=0.8),
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"))

ax.axvline(sat_95_spend / 1e6, color=CPX_RED, linewidth=1.0, linestyle=":", alpha=0.8)
ax.scatter([sat_95_spend / 1e6], [sat_95_att / 1e6], color=CPX_RED, s=100, zorder=6, marker="*")
ax.annotate(f"95% Saturation\n${sat_95_spend/1e6:.1f}M/yr\n"
            f"{sat_95_spend/year_points[2025]['annual_spend']:.1f}x 2025 spend",
            xy=(sat_95_spend / 1e6, sat_95_att / 1e6), xytext=(-80, -50), textcoords="offset points",
            fontsize=8, color=CPX_RED, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=CPX_RED, lw=0.8),
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"))

for year, style in YEAR_STYLES.items():
    pts = year_points[year]
    ax.axvline(pts["annual_spend"] / 1e6, color=style["color"], linestyle=style["linestyle"],
               linewidth=style["linewidth"], alpha=0.85, zorder=4)
    ax.scatter([pts["annual_spend"] / 1e6], [pts["annual_att"] / 1e6],
               color=style["color"], marker=style["marker"], s=90, zorder=6)
    ax.annotate(f"{year}\n${pts['annual_spend']/1e6:.1f}M/yr\n({pts['annual_att']/1e6:.1f}M att.)",
                xy=(pts["annual_spend"] / 1e6, pts["annual_att"] / 1e6),
                xytext=(10, 15 - list(YEAR_STYLES.keys()).index(year) * 40), textcoords="offset points",
                fontsize=8, color=style["color"], fontweight="bold",
                arrowprops=dict(arrowstyle="->", color=style["color"], lw=0.8),
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"))

ax.set_ylabel("Annual Media Attendance Contribution (M)", fontsize=11)
ax.set_xlabel("Annual Media Budget ($M)", fontsize=11)
ax.set_title("Total Media Response Curve\n"
             "Steady-state adstock",
             fontsize=13, fontweight="bold", color=CPX_NAVY)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}M"))
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:.1f}M"))
year_legend = [Line2D([0], [0], color=style["color"], linestyle=style["linestyle"],
                      linewidth=style["linewidth"], marker=style["marker"], markersize=7,
                      label=f"{year} (${year_points[year]['annual_spend']/1e6:.1f}M annual)")
               for year, style in YEAR_STYLES.items()]
ax.legend(handles=year_legend, fontsize=9, loc="lower right", framealpha=0.9, edgecolor=CPX_LGREY)
ax.grid(False); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.text(0.01, 0.01, "* Budget distributed using 2025 actual spend shares. Steady-state adstock applied per LOB.",
         fontsize=7, color=CPX_DGREY, fontstyle="italic", ha="left")
plt.tight_layout()
save_fig(fig, "I2_B10a_Total_Media_Response_Curve.png")
plt.show()

In [0]:
fig, ax = plt.subplots(figsize=(16, 9))
ax.plot(x_annual / 1e6, marginal_mean, color=CPX_RED, linewidth=2.5,
        label="Marginal attendees per additional $1 annual spend")
ax.axhline(0, color="black", linewidth=0.8)

if inflection_idx:
    ax.axvline(inf_spend / 1e6, color=CPX_DGREY, linewidth=1.0, linestyle="--", alpha=0.7)
    ax.annotate(f"Inflection\n${inf_spend/1e6:.1f}M/yr",
                xy=(inf_spend / 1e6, marginal_mean[inflection_idx]),
                xytext=(12, 20), textcoords="offset points",
                fontsize=8, color=CPX_DGREY, fontweight="bold",
                arrowprops=dict(arrowstyle="->", color=CPX_DGREY, lw=0.8),
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"))

ax.axvline(sat_95_spend / 1e6, color=CPX_RED, linewidth=1.0, linestyle=":", alpha=0.8)
ax.annotate(f"95% Saturation\n${sat_95_spend/1e6:.1f}M/yr",
            xy=(sat_95_spend / 1e6, marginal_mean[sat_95_idx]),
            xytext=(-80, 20), textcoords="offset points",
            fontsize=8, color=CPX_RED, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=CPX_RED, lw=0.8),
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"))

for year, style in YEAR_STYLES.items():
    pts = year_points[year]
    spend_m = pts["annual_spend"] / 1e6
    # marginal return at this year's spend — find nearest x index
    yr_idx  = np.argmin(np.abs(x_annual - pts["annual_spend"]))
    yr_marg = marginal_mean[yr_idx]
    ax.axvline(spend_m, color=style["color"], linestyle=style["linestyle"],
               linewidth=style["linewidth"], alpha=0.85, zorder=4)
    ax.scatter([spend_m], [yr_marg], color=style["color"],
               marker=style["marker"], s=90, zorder=6)
    ax.annotate(f"{year}\n${spend_m:.1f}M/yr\n({yr_marg:.2f} att/$)",
                xy=(spend_m, yr_marg),
                xytext=(10, 15 - list(YEAR_STYLES.keys()).index(year) * 40),
                textcoords="offset points",
                fontsize=8, color=style["color"], fontweight="bold",
                arrowprops=dict(arrowstyle="->", color=style["color"], lw=0.8),
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"))

ax.set_ylabel("Marginal attendees per $1 additional annual spend", fontsize=11)
ax.set_xlabel("Annual Media Spend ($M)", fontsize=11)
ax.set_title("Marginal Return Curve\n",
             fontsize=13, fontweight="bold", color=CPX_NAVY)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:.1f}M"))

year_legend = [Line2D([0], [0], color=style["color"], linestyle=style["linestyle"],
                      linewidth=style["linewidth"], marker=style["marker"], markersize=7,
                      label=f"{year} (${year_points[year]['annual_spend']/1e6:.1f}M annual)")
               for year, style in YEAR_STYLES.items()]
ax.legend(handles=year_legend, fontsize=9, loc="upper right",
          framealpha=0.9, edgecolor=CPX_LGREY)

ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.text(0.01, 0.01, "* Budget distributed using 2025 actual spend shares. Steady-state adstock applied per LOB.",
         fontsize=7, color=CPX_DGREY, fontstyle="italic", ha="left")
plt.tight_layout()
save_fig(fig, "I2_B10b_Marginal_Return_Curve.png")
plt.show()

print("\n✓ All visualizations complete.")

In [0]:
# ── CELL: Export all chart data + images to Excel ────────────────────────────
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image as XLImage
import shutil, os

NAVY  = "002B5C"; WHITE = "FFFFFF"; LGREY = "F2F2F2"; AMBER = "FFF2CC"
FUNNEL_FILLS = {"Awareness": "EEF3FA", "Consideration": "FDEAEA", "Conversion": "FFFBEA"}

hdr_font  = Font(name="Arial", bold=True, color=WHITE, size=10)
sub_font  = Font(name="Arial", bold=True, color=NAVY, size=9)
cell_font = Font(name="Arial", size=9)
center    = Alignment(horizontal="center", vertical="center", wrap_text=True)
left      = Alignment(horizontal="left",   vertical="center")
thin      = Side(style="thin", color="D0D0D0")
border    = Border(thin, thin, thin, thin)

def hfill(c=NAVY): return PatternFill("solid", fgColor=c)
def rfill(c=LGREY): return PatternFill("solid", fgColor=c)

def write_title(ws, title, subtitle, ncols):
    ws["A1"] = title
    ws["A1"].font = Font(name="Arial", bold=True, size=12, color=NAVY)
    ws.merge_cells(f"A1:{get_column_letter(ncols)}1")
    if subtitle:
        ws["A2"] = subtitle
        ws["A2"].font = Font(name="Arial", italic=True, size=9, color="555555")
        ws.merge_cells(f"A2:{get_column_letter(ncols)}2")
    ws.append([])
    return 4

def style_hdr(ws, row, cols, fcolor=NAVY):
    fill = PatternFill("solid", fgColor=fcolor)
    for j, h in enumerate(cols, 1):
        c = ws.cell(row=row, column=j, value=h)
        c.font = hdr_font; c.fill = fill
        c.alignment = center; c.border = border

def style_row(ws, row, values, fcolor=LGREY):
    fill = PatternFill("solid", fgColor=fcolor)
    for j, v in enumerate(values, 1):
        c = ws.cell(row=row, column=j, value=v)
        c.font = cell_font; c.fill = fill
        c.border = border
        c.alignment = left if j <= 2 else center

def set_widths(ws, widths):
    for j, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(j)].width = w

def embed_img(ws, path, anchor, w=900):
    if os.path.exists(path):
        img = XLImage(path)
        img.width = w; img.height = int(w * 9 / 16)
        ws.add_image(img, anchor)
    else:
        ws[anchor] = f"Image not found: {os.path.basename(path)}"

wb = Workbook()
wb.remove(wb.active)

# ════════════════════════════════════════════════════════════════════════════
# TAB 1 — Model Performance Metrics
# ════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("Model Performance")
r  = write_title(ws,
    "EXH MMM Phase 2 Iter 2 — Model Performance Metrics",
    "Train = weeks up to validation cutoff. Val = held-out validation window. "
    "MAPE = Mean Absolute % Error. R² = coefficient of determination.",
    4)

style_hdr(ws, r, ["Metric", "Train", "Validation", "Notes"]); r += 1
for label, train_v, val_v, note in [
    ("MAPE (%)",        round(mape_train, 2), round(mape_val, 2),   "Lower is better"),
    ("R²",             round(r2_train, 4),   round(r2_val, 4),     "1.0 = perfect fit"),
    ("N weeks",         n_train,              n_val,                 ""),
    ("Divergences",     0,                    0,                     "Zero divergences — clean sampler"),
]:
    style_row(ws, r, [label, train_v, val_v, note], fcolor="EEF3FA"); r += 1

r += 1
ws.cell(row=r, column=1, value="Posterior Summary — Combo-Level Parameters").font = sub_font
ws.merge_cells(f"A{r}:F{r}"); r += 1

style_hdr(ws, r, ["Combo", "Funnel", "alpha (mean)", "gamma (mean)",
                  "lam/k (mean)", "alpha_s (mean)"]); r += 1

for combo in sorted(ACTIVE_COMBOS):
    funnel = combo.split("__")[0]
    alpha  = round(float(post[f"alpha_{combo}"].values.ravel().mean()), 4)
    gamma  = round(float(post[f"gamma_{combo}"].values.ravel().mean()), 4)
    if funnel == "Consideration":
        lam_k   = round(float(post[f"k_{combo}"].values.ravel().mean()), 4)
        alpha_s = round(float(post[f"alpha_s_{combo}"].values.ravel().mean()), 4)
    else:
        lam_k   = round(float(post[f"lam_{combo}"].values.ravel().mean()), 4)
        alpha_s = "—"
    style_row(ws, r, [combo, funnel, alpha, gamma, lam_k, alpha_s],
              fcolor=FUNNEL_FILLS.get(funnel, LGREY))
    r += 1

set_widths(ws, [38, 14, 14, 14, 14, 14])
ws.freeze_panes = "A5"

# ════════════════════════════════════════════════════════════════════════════
# TAB 2 — Actual vs Predicted (weekly)
# ════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("Actual vs Predicted")
r  = write_title(ws,
    "Weekly Actual vs Predicted Attendance",
    "Segment: train / val. Residual = Actual − Predicted.",
    6)

style_hdr(ws, r, ["Week Date", "Segment", "Actual", "Predicted",
                  "Controls Only", "Residual"]); r += 1

for _, row in df_trainval.iterrows():
    seg   = row["segment"]
    fc    = "EEF5FF" if seg == "train" else "FDEAEA"
    style_row(ws, r, [
        str(row["week_date"])[:10],
        seg,
        round(row[target_col], 0),
        round(row["pred"], 0),
        round(row["controls_pred"], 0),
        round(row[target_col] - row["pred"], 0),
    ], fcolor=fc)
    r += 1

set_widths(ws, [14, 10, 14, 14, 14, 14])
ws.freeze_panes = "A5"

# ════════════════════════════════════════════════════════════════════════════
# TAB 3 — Annual Decomposition by Funnel
# ════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("Annual Decomp by Funnel")
r  = write_title(ws,
    "Annual Attendance Decomposition by Funnel",
    "Baseline = Actual − sum(media funnels). All values in attendees.",
    8)

style_hdr(ws, r, ["Year", "Actual", "Baseline (non-media)",
                  "Awareness Media", "Consideration Media", "Conversion Media",
                  "Total Media", "Media % of Actual"]); r += 1

for yr in years_avail:
    aw  = round(float(annual.loc[yr, "media_Awareness"]),     0)
    con = round(float(annual.loc[yr, "media_Consideration"]), 0)
    conv= round(float(annual.loc[yr, "media_Conversion"]),    0)
    act = round(float(annual.loc[yr, target_col]),            0)
    bl  = round(act - aw - con - conv,                        0)
    tm  = round(aw + con + conv,                              0)
    style_row(ws, r, [yr, act, bl, aw, con, conv, tm,
                      round(tm / act * 100, 1)], fcolor="EEF3FA")
    r += 1

set_widths(ws, [8, 16, 20, 18, 20, 18, 16, 18])
ws.freeze_panes = "A5"

# ════════════════════════════════════════════════════════════════════════════
# TAB 4 — Adstock Decay Parameters
# ════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("Adstock Decay")
r  = write_title(ws,
    "Adstock Decay — Funnel mu_alpha Posterior Summary",
    "Half-life = ln(0.05) / ln(alpha) weeks. 95% CI from posterior samples.",
    6)

style_hdr(ws, r, ["Funnel", "mu_alpha mean", "mu_alpha 2.5%",
                  "mu_alpha 97.5%", "Half-life (mean, wks)", "Notes"]); r += 1

for funnel in ["Awareness", "Consideration", "Conversion"]:
    samples = post[f"mu_alpha_{funnel}"].values.ravel()
    mu      = float(samples.mean())
    lo      = float(np.quantile(samples, 0.025))
    hi      = float(np.quantile(samples, 0.975))
    hl      = round(np.log(0.05) / np.log(mu), 1) if mu > 0 else "—"
    style_row(ws, r, [funnel, round(mu,4), round(lo,4), round(hi,4), hl, ""],
              fcolor=FUNNEL_FILLS.get(funnel, LGREY))
    r += 1

r += 2
ws.cell(row=r, column=1, value="Combo-Level alpha Posteriors").font = sub_font
ws.merge_cells(f"A{r}:F{r}"); r += 1
style_hdr(ws, r, ["Combo", "Funnel", "alpha mean", "alpha 2.5%",
                  "alpha 97.5%", "Half-life (wks)"]); r += 1

for combo in sorted(ACTIVE_COMBOS):
    funnel  = combo.split("__")[0]
    samples = post[f"alpha_{combo}"].values.ravel()
    mu      = float(samples.mean())
    lo      = float(np.quantile(samples, 0.025))
    hi      = float(np.quantile(samples, 0.975))
    hl      = round(np.log(0.05) / np.log(mu), 1) if mu > 0 else "—"
    style_row(ws, r, [combo, funnel, round(mu,4), round(lo,4), round(hi,4), hl],
              fcolor=FUNNEL_FILLS.get(funnel, LGREY))
    r += 1

set_widths(ws, [38, 14, 14, 14, 14, 16])
ws.freeze_panes = "A5"

# ════════════════════════════════════════════════════════════════════════════
# TAB 5 — Response Curve Data (B10a)
# ════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("Response Curve Data")
r  = write_title(ws,
    "Total Media Response Curve — Annual Budget vs Attendance",
    "Steady-state adstock. 2025 LOB budget shares held constant. "
    "89% CI from 100 posterior samples.",
    6)

style_hdr(ws, r, ["Annual Budget ($)", "Attendance (mean)",
                  "Attendance (CI low 5.5%)", "Attendance (CI high 94.5%)",
                  "Marginal att per $1", "Notes"]); r += 1

for i, b in enumerate(x_annual):
    note = ""
    if inflection_idx and i == inflection_idx:
        note = "Inflection point"
    elif i == sat_95_idx:
        note = "95% Saturation"
    elif any(abs(b - year_points[yr]["annual_spend"]) < (x_annual[1]-x_annual[0])/2
             for yr in [2023, 2024, 2025]):
        yr_match = min([2023,2024,2025], key=lambda y: abs(b - year_points[y]["annual_spend"]))
        note = f"{yr_match} actual spend"
    style_row(ws, r, [
        round(b, 0),
        round(curve_mean[i], 0),
        round(curve_lo[i], 0),
        round(curve_hi[i], 0),
        round(marginal_mean[i], 4),
        note,
    ], fcolor="EEF5FF")
    r += 1

r += 2
ws.cell(row=r, column=1, value="Year Reference Points").font = sub_font
ws.merge_cells(f"A{r}:F{r}"); r += 1
style_hdr(ws, r, ["Year", "Annual Spend ($)", "Media Attendance (att)",
                  "Marginal att/$1", "", ""]); r += 1
for yr, pts in year_points.items():
    yr_idx = np.argmin(np.abs(x_annual - pts["annual_spend"]))
    style_row(ws, r, [yr, round(pts["annual_spend"], 0),
                      round(pts["annual_att"], 0),
                      round(marginal_mean[yr_idx], 4), "", ""],
              fcolor="FFFBEA")
    r += 1

set_widths(ws, [18, 18, 18, 22, 20, 20])
ws.freeze_panes = "A5"

# ════════════════════════════════════════════════════════════════════════════
# TAB 6 — Weekly Contribution Decomp (B1 data)
# ════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("Weekly Contrib Decomp")
r  = write_title(ws,
    "Weekly Attendance Contribution Decomposition",
    "Intercept baseline + controls effect + media by funnel. All in attendees.",
    7)

style_hdr(ws, r, ["Week Date", "Segment", "Actual",
                  "Intercept Baseline", "Controls Effect",
                  "Awareness Media", "Consideration Media", "Conversion Media",
                  "Total Predicted"]); r += 1

for i, row in df_trainval.iterrows():
    seg = row["segment"]
    fc  = "EEF5FF" if seg == "train" else "FDEAEA"
    ctrl_eff = float(controls_effect[i])
    style_row(ws, r, [
        str(row["week_date"])[:10],
        seg,
        round(row[target_col], 0),
        round(baseline_orig, 0),
        round(ctrl_eff, 0),
        round(row["media_Awareness"], 0),
        round(row["media_Consideration"], 0),
        round(row["media_Conversion"], 0),
        round(row["pred"], 0),
    ], fcolor=fc)
    r += 1

set_widths(ws, [14, 10, 14, 18, 16, 16, 20, 16, 16])
ws.freeze_panes = "A5"

# ════════════════════════════════════════════════════════════════════════════
# IMAGE TABS — one per chart
# ════════════════════════════════════════════════════════════════════════════
chart_tabs = [
    ("D1 Hyperparameters",   "I2_D1_Hyperparameter_Posteriors.png"),
    ("D3 Actual vs Pred",    "I2_D3_Actual_vs_Predicted.png"),
    ("D4 Val Window",        "I2_D4_Val_Actual_vs_Predicted.png"),
    ("D5 Residuals",         "I2_D5_Residuals_Over_Time.png"),
    ("D6 Scatter",           "I2_D6_Scatter_Predicted_vs_Actual.png"),
    ("D7 Adstock Decay",     "I2_D7_Adstock_Decay_Funnel.png"),
    ("B1 Weekly Decomp",     "I2_B1_Contribution_Decomposition_Final.png"),
    ("B6 Annual Funnel",     "I2_B6_Annual_Decomposition_Funnel.png"),
    ("B10a Response Curve",  "I2_B10a_Total_Media_Response_Curve.png"),
    ("B10b Marginal Return", "I2_B10b_Marginal_Return_Curve.png"),
]

for tab_name, fname in chart_tabs:
    ws = wb.create_sheet(tab_name)
    ws["A1"] = tab_name
    ws["A1"].font = Font(name="Arial", bold=True, size=12, color=NAVY)
    embed_img(ws, os.path.join(SAVE_PATH, fname), "A3")

# ════════════════════════════════════════════════════════════════════════════
# Save
# ════════════════════════════════════════════════════════════════════════════
_tmp      = "/tmp/EXH_MMM_Diagnostics_Charts_Data.xlsx"
xlsx_path = os.path.join(SAVE_PATH, "EXH_MMM_Diagnostics_Charts_Data.xlsx")
wb.save(_tmp)
shutil.copy2(_tmp, xlsx_path)
os.remove(_tmp)
print(f"✓ Saved: {xlsx_path}")
print("  Data tabs: Model Performance | Actual vs Predicted | Annual Decomp by Funnel |")
print("             Adstock Decay | Response Curve Data | Weekly Contrib Decomp")
print(f"  Image tabs: {len(chart_tabs)} charts embedded")

In [0]:
import matplotlib.patches as mpatches

CPX_GREEN = "#2E7D32"
CPX_MID   = "#AABBCC"
FUNNELS   = ["Awareness", "Consideration", "Conversion"]

# ── Posterior mean adstock decay per funnel ────────────────────────────────
funnel_decay = {
    f: float(post[f"mu_alpha_{f}"].values.ravel().mean()) for f in FUNNELS
}
for f, d in funnel_decay.items():
    hl = np.log(0.5) / np.log(d) if 0 < d < 1 else float("inf")
    print(f"  {f}: decay = {d:.3f}  (half-life ≈ {hl:.1f} wks)")

# Optimal lag window per funnel: [0.5×half-life, 1.5×half-life], capped [1, 8]
def optimal_lag_range(decay):
    hl = max(1.0, np.log(0.5) / np.log(max(decay, 0.01)))
    return max(1, int(np.floor(hl * 0.5))), min(8, int(np.ceil(hl * 1.5)))

funnel_lags = {f: optimal_lag_range(funnel_decay[f]) for f in FUNNELS}
for f, (lmin, lmax) in funnel_lags.items():
    print(f"  {f}: optimal window = {lmin}–{lmax} wks before peak")

# ── Aggregate raw spend per funnel from df_full ──────────────────────────────
# Use spend_train_max_iter2 keys as the canonical set of spend column names
spend_cols_by_funnel = {
    f: [c for c in df_full.columns
        if c.startswith(f"{f}_") and c in spend_train_max_iter2]
    for f in FUNNELS
}

df_fl = df_full[df_full["year"] != 2026].copy()
for f in FUNNELS:
    cols = spend_cols_by_funnel[f]
    df_fl[f"spend_{f}"] = df_fl[cols].sum(axis=1) if cols else 0.0

df_fl["woy"] = df_fl["week_date"].dt.isocalendar().week.astype(int)
yrs = sorted(df_fl["year"].unique())

woy_avg = (
    df_fl.groupby("woy")
    .agg(
        attendance    =(target_col,        "mean"),
        spend_Awareness    =("spend_Awareness",    "mean"),
        spend_Consideration=("spend_Consideration", "mean"),
        spend_Conversion   =("spend_Conversion",    "mean"),
    )
    .reset_index()
)
wks = woy_avg["woy"].values

# ── Calendar zone definitions ────────────────────────────────────────────────
PEAK_WKS     = [1, 11, 33, 49, 50]
TROUGH_WKS   = list(range(19, 24)) + list(range(36, 42))
TROUGH_SPANS = [(19, 23), (36, 41)]
peak_labels  = {1: "New Year\nWeek", 11: "March\nBreak", 33: "Summer\nPeak", 49: "Pre-\nXmas"}

def get_optimal_weeks(lag_min, lag_max):
    wks_set = set()
    for p in PEAK_WKS:
        for lag in range(lag_min, lag_max + 1):
            w = p - lag
            if w <= 0: w += 52
            wks_set.add(w)
    return sorted(wks_set)

def get_opt_spans(opt_wks):
    if not opt_wks: return []
    spans, start, prev = [], opt_wks[0], opt_wks[0]
    for w in opt_wks[1:]:
        if w == prev + 1: prev = w
        else: spans.append((start, prev)); start = prev = w
    spans.append((start, prev))
    return spans

funnel_opt_wks = {f: get_optimal_weeks(*funnel_lags[f]) for f in FUNNELS}

XTICKS      = [1, 5, 9, 13, 17, 21, 26, 30, 35, 39, 43, 48, 52]
XTICKLABELS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec",""]

def clean_ax(ax):
    ax.set_xlim(1, 52); ax.set_xticks(XTICKS); ax.set_xticklabels(XTICKLABELS)
    ax.grid(False); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

FOOTER = (
    f"Note: Weekly averages {min(yrs)}–{max(yrs)}. Optimal windows from posterior mean decay. "
    + "  ".join([f"{f} decay={funnel_decay[f]:.2f}" for f in FUNNELS])
)

# ── Panel 1: Attendance calendar (all funnel windows overlaid) ────────────────
fig1, ax1 = plt.subplots(figsize=(16, 9))
for f in FUNNELS:
    for i, (s, e) in enumerate(get_opt_spans(funnel_opt_wks[f])):
        ax1.axvspan(s, e, alpha=0.13, color=FUNNEL_COLORS[f],
                   label=f"Optimal {f} window" if i == 0 else "")
for i, (s, e) in enumerate(TROUGH_SPANS):
    ax1.axvspan(s, e, alpha=0.07, color=CPX_RED, label="Attendance trough" if i == 0 else "")

ax1.plot(wks, woy_avg["attendance"] / 1e6, color="black", lw=2.5, zorder=5,
         label="Avg weekly attendance (M)")
ax1.fill_between(wks, woy_avg["attendance"] / 1e6, alpha=0.06, color="black")

for wk, lbl in peak_labels.items():
    att = woy_avg.loc[woy_avg["woy"] == wk, "attendance"].values
    if len(att):
        ax1.annotate(lbl, xy=(wk, att[0] / 1e6),
                     xytext=(wk, att[0] / 1e6 * 0.91 + 0.02),
                     ha="center", fontsize=9, color=CPX_GOLD, fontweight="bold",
                     arrowprops=dict(arrowstyle="->", color=CPX_GOLD, lw=1))

ax1.set_ylabel("Avg Weekly Attendance (M)", fontsize=11)
ax1.set_title(
    f"EXH MMM — Flighting Panel 1: Attendance Calendar\n"
    f"Avg weekly attendance by week-of-year ({min(yrs)}–{max(yrs)})  "
    f"|  Shaded = model-derived optimal spend windows per funnel",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
ax1.legend(loc="lower right", fontsize=9, frameon=True, framealpha=0.9, ncol=2)
clean_ax(ax1)
fig1.text(0.5, -0.02, FOOTER, ha="center", fontsize=7.5, color=CPX_DGREY, style="italic")
plt.tight_layout()
save_fig(fig1, "EXH_flighting_p1_attendance_calendar.png")
plt.show()

# ── Panels 2-4: One per funnel ─────────────────────────────────────────────────
flighting_rows = []
for funnel in FUNNELS:
    spend_key  = f"spend_{funnel}"
    color      = FUNNEL_COLORS[funnel]
    lag_min, lag_max = funnel_lags[funnel]
    opt_wks    = funnel_opt_wks[funnel]

    sp_total   = woy_avg[spend_key].sum()
    sp_opt     = woy_avg.loc[woy_avg["woy"].isin(opt_wks),       spend_key].sum()
    sp_trough  = woy_avg.loc[woy_avg["woy"].isin(TROUGH_WKS),    spend_key].sum()
    sp_peak    = woy_avg.loc[woy_avg["woy"].isin(PEAK_WKS),      spend_key].sum()
    sp_other   = sp_total - sp_opt - sp_trough - sp_peak

    flighting_rows.append({
        "Funnel":            funnel,
        "Decay":             round(funnel_decay[funnel], 3),
        "Optimal lag (wks)": f"{lag_min}–{lag_max}",
        "Optimal %":         round(sp_opt    / sp_total * 100, 1) if sp_total else 0,
        "Trough %":          round(sp_trough / sp_total * 100, 1) if sp_total else 0,
        "Peak-wk % (late)":  round(sp_peak   / sp_total * 100, 1) if sp_total else 0,
        "Neutral %":         round(sp_other  / sp_total * 100, 1) if sp_total else 0,
    })

    bar_colors = [
        CPX_RED  if w in TROUGH_WKS else
        CPX_GOLD if w in PEAK_WKS   else
        color    if w in opt_wks     else
        CPX_MID
        for w in wks
    ]

    fig_f, ax_f = plt.subplots(figsize=(16, 9))
    ax_f.bar(wks, woy_avg[spend_key] / 1e3, color=bar_colors, alpha=0.85, width=0.85)
    for i, (s, e) in enumerate(get_opt_spans(opt_wks)):
        ax_f.axvspan(s, e, alpha=0.09, color=color, zorder=0)
    for i, (s, e) in enumerate(TROUGH_SPANS):
        ax_f.axvspan(s, e, alpha=0.06, color=CPX_RED, zorder=0)

    ymax = woy_avg[spend_key].max() / 1e3 if woy_avg[spend_key].max() > 0 else 1
    if sp_total > 0:
        ax_f.text(7,  ymax * 0.88, f"Optimal\n{sp_opt/sp_total*100:.0f}%",
                  ha="center", fontsize=9, fontweight="bold", color=color,
                  bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=color, alpha=0.92))
        ax_f.text(38, ymax * 0.88, f"Trough\n{sp_trough/sp_total*100:.0f}%",
                  ha="center", fontsize=9, fontweight="bold", color=CPX_RED,
                  bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=CPX_RED, alpha=0.92))

    ax_f.set_ylabel("Avg Weekly Spend ($K)", fontsize=11)
    ax_f.set_title(
        f"EXH MMM — Flighting: {funnel} Spend by Zone\n"
        f"Posterior mean decay = {funnel_decay[funnel]:.3f}  "
        f"|  Optimal window = {lag_min}–{lag_max} wks before attendance peak",
        fontsize=13, fontweight="bold", color=CPX_NAVY,
    )
    ax_f.legend(
        handles=[
            mpatches.Patch(color=color,    alpha=0.85, label=f"Optimal window — {sp_opt/sp_total*100:.0f}% of spend"  if sp_total else "Optimal window"),
            mpatches.Patch(color=CPX_RED,  alpha=0.85, label=f"Attendance trough — {sp_trough/sp_total*100:.0f}% (low return)" if sp_total else "Trough"),
            mpatches.Patch(color=CPX_GOLD, alpha=0.85, label=f"Peak week itself — {sp_peak/sp_total*100:.0f}% (too late)"    if sp_total else "Peak wk"),
            mpatches.Patch(color=CPX_MID,  alpha=0.85, label=f"Neutral weeks — {sp_other/sp_total*100:.0f}%"                if sp_total else "Neutral"),
        ],
        loc="upper left", fontsize=10, frameon=True, framealpha=0.9, ncol=2,
    )
    clean_ax(ax_f)
    fig_f.text(0.5, -0.02, FOOTER, ha="center", fontsize=7.5, color=CPX_DGREY, style="italic")
    plt.tight_layout()
    save_fig(fig_f, f"EXH_flighting_{funnel.lower()}_spend_zones.png")
    plt.show()

# ── Summary table ──────────────────────────────────────────────────────────────
print("\nEXH Flighting Analysis — Spend Zone Breakdown by Funnel:")
display(pd.DataFrame(flighting_rows))


In [0]:
# ── Per-LOB flighting zone breakdown ────────────────────────────────────────────────
# Scans ALL {Funnel}__{LOB} columns in df_full (original LOBs, before any merging).
# LOB label strips the funnel prefix — e.g. "Awareness__Premium_Brand" → "Premium Brand".
all_lob_cols = sorted([
    c for c in df_full.columns
    if any(c.startswith(f"{f}__") for f in FUNNELS)
])
print(f"Found {len(all_lob_cols)} original LOB columns in df_full:")
for c in all_lob_cols:
    print(f"  {c}")

lob_rows = []

for combo in all_lob_cols:
    funnel = next(f for f in FUNNELS if combo.startswith(f"{f}__"))
    if combo not in df_fl.columns:
        continue

    lob_label = combo.split("__", 1)[1].replace("_", " ")
    opt_wks   = funnel_opt_wks[funnel]
    lag_min, lag_max = funnel_lags[funnel]

    woy_lob = (
        df_fl.groupby("woy")[combo]
        .mean()
        .reset_index()
        .rename(columns={combo: "spend"})
    )

    total = woy_lob["spend"].sum()
    if total == 0:
        continue

    sp_opt    = woy_lob.loc[woy_lob["woy"].isin(opt_wks),    "spend"].sum()
    sp_trough = woy_lob.loc[woy_lob["woy"].isin(TROUGH_WKS), "spend"].sum()
    sp_peak   = woy_lob.loc[woy_lob["woy"].isin(PEAK_WKS),   "spend"].sum()
    sp_other  = total - sp_opt - sp_trough - sp_peak

    lob_rows.append({
        "LOB":                  lob_label,
        "Optimal %":            round(sp_opt    / total * 100, 1),
        "Neutral %":            round(sp_other  / total * 100, 1),
        "Peak wk % (too late)": round(sp_peak   / total * 100, 1),
        "Trough %":             round(sp_trough / total * 100, 1),
        "Avg wkly spend $K":    round(woy_lob["spend"].mean() / 1e3, 2),
        "Optimal lag (wks)":    f"{lag_min}–{lag_max}",
    })

lob_df = (
    pd.DataFrame(lob_rows)
    .sort_values("Optimal %", ascending=False)
    .reset_index(drop=True)
)

print("LOB Spend Zone Breakdown (each LOB uses its funnel-specific optimal window):")
display(lob_df)

# ── Chart: horizontal stacked bar, sorted by Optimal % ──────────────────────────
# Reverse order so highest optimal % is at the top
plot_df = lob_df.iloc[::-1].reset_index(drop=True)
n = len(plot_df)
y = np.arange(n)

fig_h = max(9, n * 0.55 + 2)  # scale height to number of LOBs
fig, ax = plt.subplots(figsize=(16, fig_h))
lefts = np.zeros(n)

for label, color in [
    ("Optimal",           CPX_GREEN),
    ("Neutral",           CPX_MID),
    ("Peak wk % (too late)", CPX_GOLD),
    ("Trough %",          CPX_RED),
]:
    col_key = label if label in plot_df.columns else \
              "Peak wk % (too late)" if "Peak" in label else "Trough %"
    # Map display label to column
    col_map = {
        "Optimal":               "Optimal %",
        "Neutral":               "Neutral %",
        "Peak wk % (too late)": "Peak wk % (too late)",
        "Trough %":              "Trough %",
    }
    vals = plot_df[col_map[label]].values
    ax.barh(y, vals, left=lefts, height=0.6, color=color, alpha=0.88, label=label)
    for i, (v, l) in enumerate(zip(vals, lefts)):
        if v >= 6:
            txt_color = "white" if color not in (CPX_MID, CPX_GOLD) else CPX_DGREY
            ax.text(l + v / 2, i, f"{v:.0f}%",
                    ha="center", va="center", fontsize=8.5, fontweight="bold",
                    color=txt_color)
    lefts += vals

# Avg spend annotation on right edge
for i, row in plot_df.iterrows():
    ax.text(102, i, f"${row['Avg wkly spend $K']:.1f}K/wk",
            va="center", fontsize=8, color=CPX_DGREY)

ax.axvline(100, color="#DDDDDD", lw=1)
ax.set_yticks(y)
ax.set_yticklabels(plot_df["LOB"].tolist(), fontsize=10)
ax.set_xlabel("Share of Average Weekly Spend (%)", fontsize=11)
ax.set_title(
    "EXH MMM — Original LOB Spend Zone Breakdown\n"
    "Sorted by % landing in optimal pre-peak window  |  Right axis = avg weekly spend",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
ax.legend(loc="lower right", fontsize=10, frameon=True, framealpha=0.9)
ax.set_xlim(0, 120)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "EXH_flighting_lob_zone_breakdown.png")
plt.show()
